#### Investigating the outliers in rf1, rf2, and sf2 datasets

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
import numpy as np
import matplotlib.pyplot as plt
from moc.metrics.distribution_metrics import pce, multivariate_energy_score, mse
import torch

In [3]:
import wandb
wandb.login(key="9d338bfb8d6dd9ab97384ee89b11f332ae3e12b8") #ELNURA'S KEY

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: ryuzaki to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
config = get_config()
config.device = 'cuda'
data_group, data_name = 'mulan', 'rf1'
hparams = {
    'model': 'mixture',
    'prerank': 'none',
    'lambda': 0.0,
}

In [9]:
rc = RunConfig(config, data_group, data_name, hparams = hparams)
datamodule = RealDataModule(rc, num_workers=8)
p, q = datamodule.input_dim, datamodule.output_dim #268,16

In [10]:
model = MixtureLightningModule(p, q)
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
wandb.finish()

/mnt/default/elnura_workspace/Multivariate-recalibration/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/default/elnura_workspace/Multivariate-recalibra ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]


Checking False, False


epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/marg_val,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/nll,█▇▇▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
train/prerank_val,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total_loss,█▇▇▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
val/nll,▃▂▁▁▁▂▆▇█▆▃▇▃▂▂▄▁▂▂
epoch,18
train/marg_val,0
train/nll,3.11868
train/prerank_val,0


In [12]:
ckpt_path = trainer.checkpoint_callback.best_model_path
best_model = MixtureLightningModule.load_from_checkpoint(ckpt_path)
best_model.eval().to(config.device)
test_loader = datamodule.test_dataloader()

for x, y, idx in test_loader:
    x = x.to(config.device)
    y = y.to(config.device)
    dist = best_model.predict(x)
    nll_value = -dist.log_prob(y)
    print(nll_value.mean().item())

9.564727783203125
9.137224197387695
8.998113632202148
9.04083251953125
9.392816543579102
9.54735279083252
144.4379425048828
9.009527206420898
